# EOQ Lab — National CRDC & Ed-Tech Data Collection

**Focus areas:**
1. **Civil Rights Data Collection (CRDC)** — expand beyond 2017–18
2. **Technology in schools** — internet, devices, digital learning

**Prerequisite:** Run `collect_education_data.ipynb` first (Phase 4 CRDC setup). This notebook **adds** new federal files and logs to the same `logs/manifest.csv`.

Run cells **top to bottom**.


In [1]:
%pip install -q requests beautifulsoup4 pandas openpyxl


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\saihaj\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


---
## Step 0 — Project paths & imports


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from federal_collect import (
    CRDC_PUBLIC_ZIPS,
    crdc_api_dest_filename,
    discover_crdc_api_files,
    discover_edtech_files,
    download_to_path,
    edtech_dest_filename,
    ensure_federal_layout,
    inventory_crdc_zips,
    list_crdc_zip_topics,
    refresh_manifest,
)

PATHS = ensure_federal_layout(PROJECT_ROOT)
DOWNLOAD_LOG = PATHS["logs_dir"] / "download_log.jsonl"
MANIFEST_CSV = PATHS["logs_dir"] / "manifest.csv"
FEDERAL_DISCOVERED = PATHS["logs_dir"] / "federal_phase2_discovered_links.csv"

print("Project root:", PROJECT_ROOT.resolve())
print("CRDC folder:", PATHS["crdc_root"].resolve())
print("Ed-tech folder:", PATHS["edtech_root"].resolve())


Project root: C:\Users\saihaj\Documents\Data Collection - EOQ Lab
CRDC folder: C:\Users\saihaj\Documents\Data Collection - EOQ Lab\data\raw\federal\crdc
Ed-tech folder: C:\Users\saihaj\Documents\Data Collection - EOQ Lab\data\raw\federal\edtech


---
## Phase 1 — Download CRDC public-use zip bundles (national)

These zips contain **all U.S. states** — school and LEA CSV files.

| Year | Approx. size | Default |
|------|-------------|---------|
| 2015–16 | ~31 MB | Yes |
| 2017–18 | ~114 MB | Skip if you already downloaded in the first notebook |
| 2020–21 | ~78 MB | Yes (newer + more tech fields) |
| 2021–22 | ~800 MB | **Optional** (set flag below) |


In [2]:
# --- Phase 1 settings ---
SKIP_YEARS_ALREADY_PRESENT = True   # skip zip if file already on disk
DOWNLOAD_2017_18 = False            # you likely have this from the first notebook
DOWNLOAD_2021_22 = False            # very large; turn on only if you want the newest CRDC

YEARS_TO_DOWNLOAD = {"2015-16", "2020-21"}
if DOWNLOAD_2017_18:
    YEARS_TO_DOWNLOAD.add("2017-18")
if DOWNLOAD_2021_22:
    YEARS_TO_DOWNLOAD.add("2021-22")

phase1_records = []

for item in CRDC_PUBLIC_ZIPS:
    year = item["year"]
    if year not in YEARS_TO_DOWNLOAD:
        print(f"SKIP (not selected): {year}")
        continue

    dest = PATHS["crdc_root"] / year / item["filename"]
    if SKIP_YEARS_ALREADY_PRESENT and dest.exists() and dest.stat().st_size > 0:
        print(f"Already have: {dest.relative_to(PROJECT_ROOT)}")
        continue

    print(f"Downloading {year} CRDC zip... (may take several minutes)")
    record = download_to_path(
        item["url"],
        dest,
        description=f"CRDC public-use zip {year}",
        category="discipline",
        download_log=DOWNLOAD_LOG,
        project_root=PROJECT_ROOT,
        timeout_seconds=900,
    )
    record["local_path"] = str(dest.relative_to(PROJECT_ROOT))
    phase1_records.append(record)
    size_mb = record["bytes"] / 1024 / 1024
    print(f"  -> {record['status']}: {dest.name} ({size_mb:.1f} MB)")

if phase1_records:
    display(pd.DataFrame(phase1_records))
else:
    print("No new CRDC zips downloaded.")


Already have: data\raw\federal\crdc\2015-16\2015-16-crdc-data.zip
SKIP (not selected): 2017-18
Already have: data\raw\federal\crdc\2020-21\2020-21-crdc-data.zip
SKIP (not selected): 2021-22
No new CRDC zips downloaded.


---
## Phase 2 — Inventory CRDC topics (incl. technology)

Lists every CSV topic inside each CRDC zip. Topics matching **internet / computer / device / digital / distance** are flagged.


In [3]:
inventory_df = inventory_crdc_zips(PATHS["crdc_root"])
display(inventory_df)

TECH_KEYS = ("internet", "wifi", "computer", "device", "technology", "digital", "distance", "broadband")

for zip_path in sorted(PATHS["crdc_root"].rglob("*-crdc-data.zip")):
    year = zip_path.parent.name
    topics = list_crdc_zip_topics(zip_path)
    tech = [t for t in topics if any(k in t.lower() for k in TECH_KEYS)]
    print(f"\n{year}: {len(tech)} tech-related topics (of {len(topics)} total)")
    for t in tech:
        print(f"  - {t}")


,year,zip_file,topics,tech_topics,tech_topic_names,error
0,2015-16,2015-16-crdc-data.zip,4,0,,
1,2020-21,2020-21-crdc-data.zip,35,3,Computer Science; Distance Education; Internet...,



2015-16: 0 tech-related topics (of 4 total)

2020-21: 3 tech-related topics (of 35 total)
  - Computer Science
  - Distance Education
  - Internet Access and Devices


---
## Phase 3 — CRDC topic files from data.ed.gov (2015–16 & 2020–21)

The API gives **individual** CRDC spreadsheets (discipline, enrollment, etc.) — complements the zip bundles.


In [4]:
crdc_api_df = discover_crdc_api_files()
print(f"CRDC API file links found: {len(crdc_api_df)}")
crdc_api_df.to_csv(FEDERAL_DISCOVERED, index=False)
crdc_api_df.head(10)


CRDC API file links found: 189


,source,year_hint,category,dataset_title,format,url,description
0,CRDC,2015-16,discipline,2015-16 Discipline Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2015-16 Discipline Civil Rights Data Col...
1,CRDC,2015-16,discipline,2015-16 Discipline Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2015-16 Discipline Civil Rights Data Col...
2,CRDC,2015-16,discipline,2015-16 Discipline Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2015-16 Discipline Civil Rights Data Col...
3,CRDC,2015-16,discipline,2015-16 Discipline Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2015-16 Discipline Civil Rights Data Col...
4,CRDC,2015-16,discipline,2015-16 Discipline Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2015-16 Discipline Civil Rights Data Col...
5,CRDC,2015-16,discipline,2015-16 Discipline Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2015-16 Discipline Civil Rights Data Col...
6,CRDC,2015-16,discipline,2015-16 Discipline Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2015-16 Discipline Civil Rights Data Col...
7,CRDC,2015-16,discipline,2015-16 Discipline Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2015-16 Discipline Civil Rights Data Col...
8,CRDC,2015-16,discipline,2015-16 Discipline Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2015-16 Discipline Civil Rights Data Col...
9,CRDC,2015-16,discipline,2015-16 Discipline Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2015-16 Discipline Civil Rights Data Col...


In [5]:
# Download CRDC API files into data/raw/federal/{category}/
# (ArcGIS map links are excluded; failed URLs are logged and skipped — loop keeps going)
TARGET_API_YEARS = ("2015-16", "2015-2016", "2020-21", "2020-2021")

api_records = []
for _, row in crdc_api_df.iterrows():
    hint = str(row.get("year_hint", ""))
    blob = f"{hint} {row['dataset_title']} {row['url']}"
    if not any(y in blob for y in TARGET_API_YEARS):
        continue

    fname = crdc_api_dest_filename(row)
    dest_path = PATHS["federal_root"] / row["category"] / fname
    print(f"[{len(api_records)+1}] {row['dataset_title'][:70]}...")
    record = download_to_path(
        row["url"],
        dest_path,
        description=row["description"],
        category=row["category"],
        download_log=DOWNLOAD_LOG,
        project_root=PROJECT_ROOT,
    )
    if record["status"] == "failed":
        print(f"  -> FAILED: {record.get('error', 'unknown')[:100]}")
    elif record["status"] == "skipped_exists":
        print(f"  -> skipped (already had file): {fname}")
    else:
        print(f"  -> downloaded: {fname}")
    record["local_path"] = str(dest_path.relative_to(PROJECT_ROOT))
    api_records.append(record)

print(f"\nCRDC API downloads attempted: {len(api_records)}")
downloaded = sum(1 for r in api_records if r["status"] == "downloaded")
skipped = sum(1 for r in api_records if r["status"] == "skipped_exists")
failed = sum(1 for r in api_records if r["status"] == "failed")
print(f"New downloads: {downloaded}, skipped: {skipped}, failed: {failed}")


[1] 2015-16 Discipline Civil Rights Data Collection...
  -> downloaded: 2015-16_Preschool-Discipline.xlsx
[2] 2015-16 Discipline Civil Rights Data Collection...
  -> downloaded: 2015-16_Preschool-Expulsions.xlsx
[3] 2015-16 Discipline Civil Rights Data Collection...
  -> downloaded: 2015-16_Preschool-One-or-More-out-of-school-suspensions.xlsx
[4] 2015-16 Discipline Civil Rights Data Collection...
  -> downloaded: 2015-16_Preschool-More-than-one-out-of-school-suspension.xlsx
[5] 2015-16 Discipline Civil Rights Data Collection...
  -> downloaded: 2015-16_Preschool-One-out-of-school-suspension.xlsx
[6] 2015-16 Discipline Civil Rights Data Collection...
  -> downloaded: 2015-16_Preschool-Corporal-Punishment.xlsx
[7] 2015-16 Discipline Civil Rights Data Collection...
  -> downloaded: 2015-16_Days-missed-due-to-out-of-school-suspensions.xlsx
[8] 2015-16 Discipline Civil Rights Data Collection...
  -> downloaded: 2015-16_Transferred-to-alternative-school.xlsx
[9] 2015-16 Discipline Civil Righ

---
## Phase 4 — NCES / ED ed-tech survey downloads

National surveys on **technology in schools**, **internet access**, and **teacher technology use**.


In [2]:
edtech_df = discover_edtech_files()
print(f"Ed-tech file links found: {len(edtech_df)}")
edtech_df.head(10)


Ed-tech file links found: 52


,source,category,dataset_title,format,url,description
0,NCES/FRSS,other,"Educational Technology in Public Schools, 2008",ZIP,https://nces.ed.gov/surveys/frss/download/data...,Ed-tech: Educational Technology in Public Scho...
1,NCES/FRSS,other,Educational Technology in Public School Distri...,ZIP,https://nces.ed.gov/surveys/frss/download/data...,Ed-tech: Educational Technology in Public Scho...
2,NCES/FRSS,other,"Internet Access in U.S. Public Schools, 2003",ZIP,https://nces.ed.gov/surveys/frss/download/data...,Ed-tech: Internet Access in U.S. Public School...
3,NCES/FRSS,other,Teachers' Use of Educational Technology in U.S...,ZIP,https://nces.ed.gov/surveys/frss/download/data...,Ed-tech: Teachers' Use of Educational Technolo...
4,NCES/ED,other,"Educational Technology in Public Schools, 2008",ZIPPED SAS,https://nces.ed.gov/surveys/frss/download/data...,Ed-tech: Educational Technology in Public Scho...
5,NCES/ED,other,Educational Technology in Public School Distri...,ZIPPED SAS,https://nces.ed.gov/surveys/frss/download/data...,Ed-tech: Educational Technology in Public Scho...
6,NCES/ED,other,Teachers' Use of Educational Technology in U.S...,ZIPPED SAS,https://nces.ed.gov/surveys/frss/download/data...,Ed-tech: Teachers' Use of Educational Technolo...
7,NCES/ED,other,"Internet Access in U.S. Public Schools, 2003",ZIPPED SAS,https://nces.ed.gov/surveys/frss/download/data...,Ed-tech: Internet Access in U.S. Public School...
8,NCES/ED,other,Distance Education Courses for Public Elementa...,ZIPPED DAT,https://nces.ed.gov/surveys/frss/download/data...,Ed-tech: Distance Education Courses for Public...
9,NCES/ED,other,Distance Education Courses for Public Elementa...,ZIPPED SAS,https://nces.ed.gov/surveys/frss/download/data...,Ed-tech: Distance Education Courses for Public...


In [3]:
edtech_records = []
for i, row in edtech_df.iterrows():
    fname = edtech_dest_filename(row)
    dest_path = PATHS["edtech_root"] / fname
    print(f"[{i+1}/{len(edtech_df)}] {row['dataset_title'][:65]}...")
    record = download_to_path(
        row["url"],
        dest_path,
        description=row["description"],
        category="other",
        download_log=DOWNLOAD_LOG,
        project_root=PROJECT_ROOT,
    )
    if record["status"] == "failed":
        print(f"  -> FAILED: {record.get('error', 'unknown')[:100]}")
    elif record["status"] == "skipped_exists":
        print(f"  -> skipped (already had file): {fname}")
    else:
        print(f"  -> downloaded: {fname}")
    record["local_path"] = str(dest_path.relative_to(PROJECT_ROOT))
    edtech_records.append(record)

print(f"\nEd-tech downloads attempted: {len(edtech_records)}")
downloaded = sum(1 for r in edtech_records if r["status"] == "downloaded")
skipped = sum(1 for r in edtech_records if r["status"] == "skipped_exists")
failed = sum(1 for r in edtech_records if r["status"] == "failed")
print(f"New downloads: {downloaded}, skipped: {skipped}, failed: {failed}")


[1/52] Educational Technology in Public Schools, 2008...
  -> downloaded: f92data.zip
[2/52] Educational Technology in Public School Districts, 2008...
  -> downloaded: f93data.zip
[3/52] Internet Access in U.S. Public Schools, 2003...
  -> downloaded: f86data.zip
[4/52] Teachers' Use of Educational Technology in U.S. Public Schools, 2...
  -> downloaded: f95data.zip
[5/52] Educational Technology in Public Schools, 2008...
  -> downloaded: f92sas.zip
[6/52] Educational Technology in Public School Districts, 2008...
  -> downloaded: f93sas.zip
[7/52] Teachers' Use of Educational Technology in U.S. Public Schools, 2...
  -> downloaded: f95sas.zip
[8/52] Internet Access in U.S. Public Schools, 2003...
  -> downloaded: f86sas.zip
[9/52] Distance Education Courses for Public Elementary and Secondary Sc...
  -> downloaded: f89data.zip
[10/52] Distance Education Courses for Public Elementary and Secondary Sc...
  -> downloaded: f89sas.zip
[11/52] Distance Education Courses for Public Elementa

---
## Phase 5 — Refresh manifest & summary


In [4]:
# Works on its own if you already ran Phases 1-4 — just run Step 0 first if kernel was restarted
if "PATHS" not in globals():
    raise NameError("Run Step 0 (imports) first, then re-run this cell.")

manifest = refresh_manifest(DOWNLOAD_LOG, MANIFEST_CSV)

fed = manifest[manifest["state"] == "federal"] if not manifest.empty else pd.DataFrame()

summary_rows = [
    {"location": "data/raw/federal/crdc/", "files": sum(1 for p in PATHS["crdc_root"].rglob("*") if p.is_file())},
    {"location": "data/raw/federal/edtech/", "files": sum(1 for p in PATHS["edtech_root"].rglob("*") if p.is_file())},
    {"location": "data/raw/federal/ (all)", "files": sum(1 for p in PATHS["federal_root"].rglob("*") if p.is_file())},
]

print("=== Collection summary ===")
print(pd.DataFrame(summary_rows).to_string(index=False))

if not fed.empty:
    print("\nFederal manifest rows by category:")
    print(fed.groupby("category").size())

print(f"\nManifest refreshed: {MANIFEST_CSV}")


=== Collection summary ===
                location  files
  data/raw/federal/crdc/      2
data/raw/federal/edtech/     46
 data/raw/federal/ (all)    347

Federal manifest rows by category:
category
discipline      79
enrollment      35
other          232
teachers        10
test_scores     32
dtype: int64

Manifest refreshed: c:\Users\saihaj\Documents\Data Collection - EOQ Lab\logs\manifest.csv
